[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ElenaVillano/prope-programacion/blob/main/materiales/m08_visualizacion.ipynb)

# Propedéutico de Programación para el Análisis de datos

**EGobiernoyTP, verano 2026**

## Análisis exploratorio y visualización de datos

## Contexto de la base de datos

- **Fuente:** National Center for Health Statistics (NCHS) + U.S. Census Bureau
- **Qué es:** Encuesta experimental en línea (~20 min) lanzada en abril 2020 por el Census Bureau junto con NCHS y otras agencias federales, para medir en tiempo casi real el impacto de la pandemia de COVID-19 en los hogares estadounidenses.
- La base mide **acceso y necesidad de atención de salud mental** entre adultos en Estados Unidos, a lo largo de distintos periodos de levantamiento.
- No son registros individuales: cada fila contiene una **estimación agregada (porcentaje)** para una población y periodo determinados.

#### Diccionario de datos





| Columna | Descripción |
|---|---|
| Indicator | Indicador medido (ej. "Symptoms of Anxiety Disorder") |
| Group | Categoría de desagregación (National Estimate, By Age, By Sex, By State, By Race/Hispanic ethnicity...) |
| State | Estado o "United States" (nacional) |
| Subgroup | Valor específico dentro del Group (ej. "18-29 years") |
| Phase | Fase de la encuesta (1, 2, 3, 3.1...) |
| Time Period | N° secuencial de la ronda de recolección |
| Time Period Label | Rango de fechas legible (ej. "Aug 19 - Aug 31, 2020") |
| Time Period Start Date | Fecha de inicio del periodo |
| Time Period End Date | Fecha de fin del periodo |
| Value | Estimación (%) |
| LowCI | Límite inferior del IC 95% |
| HighCI | Límite superior del IC 95% |
| Confidence Interval | IC en texto (ej. "19.0 - 19.8") |
| Quartile Range | Rango de cuartil (indicadores de vivienda/gasto) |
| Suppression Flag | Marca de supresión por baja fiabilidad estadística |

- Un renglón:
  
  ```
  Indicator: Took Prescription Medication for Mental Health, Last 4 Weeks
  Group: By Age
  State: United States
  Subgroup: 18 - 29 years
  Phase: 2
  Time Period: 13
  Time Period Label: Aug 19 - Aug 31, 2020
  Value: 18.7
  LowCI: 17.2
  HighCI: 20.3
  ```

- Se interpreta de la siguiente manera: durante ese periodo, se estimó que 18.7% de los adultos estadounidenses de 18 a 29 años habían tomado medicamentos recetados para su salud mental en las últimas cuatro semanas, con un intervalo de confianza de 17.2% a 20.3%.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


# Estilo base para todas las gráficas del notebook
# 'whitegrid' agrega líneas horizontales de referencia — ayudan a leer valores
sns.set_theme(style='whitegrid', palette='crest')

# Tamaño por defecto de las figuras (ancho x alto en pulgadas)
plt.rcParams['figure.figsize'] = (10, 5)

# Fuente más grande para que se lea bien en pantalla y en proyector
plt.rcParams['font.size'] = 12

In [ ]:
# Para ver 100 filas en el notebook
pd.set_option('display.max_rows', 100)
#Para ver 10 columnas en el notebook
pd.set_option('display.max_columns', 10)

In [ ]:
# Carga de datos
df = pd.read_csv('https://raw.githubusercontent.com/ElenaVillano/prope-programacion/refs/heads/main/data/datos.csv')
df.head()

## Limpieza

In [ ]:
# Hacemos todas las columnas con minúculas y reemplazamos espacios por _
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

In [ ]:
# Solo nos quedamos con las columnas que analizaremos
df.drop(['time_period_label', 'confidence_interval','time_period',
         'quartile_range','suppression_flag'], inplace=True, axis=1)

In [ ]:
# Convertimos las variables de tiempo
df['time_period_start_date'] = pd.to_datetime(df['time_period_start_date'], errors='coerce')
df['time_period_end_date'] = pd.to_datetime(df['time_period_end_date'], errors='coerce')

In [ ]:
df.dtypes

## Análisis exploratorio descriptivo

In [ ]:
df['indicator'].value_counts()

In [ ]:
# Cambiamos el nombre de las etiquitas para facilidad de lectura
df['indicator'] = df['indicator'].replace({'Received Counseling or Therapy, Last 4 Weeks':
                            'Recibió terapia',
                         'Needed Counseling or Therapy But Did Not Get It, Last 4 Weeks':
                            'Necesitó terapia pero no la obtuvo',
                         'Took Prescription Medication for Mental Health, Last 4 Weeks':
                            'Tomó medicamentos para la salud mental',
                         'Took Prescription Medication for Mental Health And/Or Received Counseling or Therapy, Last 4 Weeks':
                            'Tomó medicamentos para la salud mental y/o recibió terapia'
                         })

In [ ]:
df['indicator'].value_counts()

In [ ]:
# El valor de la media de todos los indicadores combinados.
df[(df['group']!='By State') & (df['group']!='National Estimate')].groupby(["indicator","group", "subgroup"])["value"].mean().head(20)

## Análisis visual

#### Histograma

Distribución de frecuencias

In [ ]:
g = sns.FacetGrid(df.dropna(subset=['value', 'indicator']),
                  col='indicator',
                  col_wrap=2,
                  height=4,
                  aspect=1.2,
                  sharex=True, sharey=True)
g.map(plt.hist, 'value', bins=30, edgecolor='k')
g.set_titles("{col_name}")
g.set_axis_labels('Value', 'Frecuencia')
plt.show()

#### Boxplot o diagrama de caja

Un **boxplot** resume la distribución y permite observar la mediana, los cuartiles, la dispersión y posibles valores atípicos.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=df.dropna(subset=['value', 'indicator']),
    y='indicator', # ¿Qué pasa si el bimestre va en el eje y?
    x='value',
    palette='Set2',
    hue='indicator',
    legend=False,  # ¿Qué pasa si le pones True?
    ax=ax
)

ax.set_title('Distribución del valor por indicador', fontsize=13)
ax.set_xlabel('Porcentaje estimado de personas (%)')
ax.set_ylabel('Indicador', fontsize=11)

plt.tight_layout()
plt.show()

#### Scatterplot o gráfico de dispersión

Se usa para la relación entre dos variables numéricas

In [ ]:
plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=df,
    x='lowci',
    y='highci',
    alpha=0.5
)

plt.xlabel('Límite inferior (LowCI)')
plt.ylabel('Límite superior (HighCI)')
plt.title('Relación entre los límites del intervalo de confianza')

plt.show()

### Nota sobre `Group`, `Subgroup` y `Value`

Antes de analizar la variable `Value`, es importante entender qué representa cada observación de la base.

La variable `Value` contiene el **porcentaje estimado de personas que presentan el indicador correspondiente**. Sin embargo, las filas de la base no representan personas individuales.

Las variables `Group` y `Subgroup` indican diferentes formas de desagregar la población. Por ejemplo, podemos encontrar estimaciones para:

* la población nacional (`National Estimate`);
* distintos grupos de edad (`By Age`);
* niveles educativos (`By Education`);
* estados (`By State`);
* presencia de síntomas de ansiedad o depresión, entre otros.

Por lo tanto, estas observaciones **no son observaciones independientes de distintas personas**. Son estimaciones del mismo fenómeno realizadas para diferentes cortes o subpoblaciones.

Esto significa que no es conveniente interpretar conjuntamente todos los valores de `Value` como si fueran observaciones equivalentes. Por ejemplo, una estimación nacional puede coexistir con estimaciones por edad, educación o estado para un mismo periodo. Si las mezclamos en una sola distribución, estaremos combinando distintos niveles de desagregación de la población.

Por esta razón, para algunos análisis resulta más adecuado **seleccionar primero un `Group` específico** y después comparar sus `Subgroup`. Por ejemplo, podemos analizar por separado los grupos de edad, los niveles educativos o los estados.

Esta estructura también es importante para interpretar los gráficos: una barra, punto o valor no representa el número de personas en la base, sino una **estimación porcentual correspondiente a una población o subpoblación determinada**.

##### Hacerlo solo por el estimado Nacional



In [ ]:
df_nacional = df[df['group'] == 'National Estimate']

In [ ]:
df_nacional

##### Histograma National Estimate

In [ ]:
g = sns.FacetGrid(
    df_nacional.dropna(subset=['value', 'indicator']),
    col='indicator',
    col_wrap=2,
    height=4,
    aspect=1.2,
    sharex=True,
    sharey=True
)

g.map(plt.hist, 'value', bins=20, edgecolor='k')

g.set_axis_labels(
    'Porcentaje estimado de personas (%)',
    'Número de periodos'
)
g.set_titles("{col_name}")
plt.show()

##### Boxplot National Estimate

In [ ]:
sns.boxplot(
    data=df_nacional.dropna(subset=['value', 'indicator']),
    y='indicator',
    x='value',
    hue='indicator',
    legend=False
)

plt.xlabel('Porcentaje estimado de personas (%)')
plt.ylabel('Indicador')
plt.title('Distribución de las estimaciones nacionales por indicador')

plt.show()

#### Gráfico de barras

Los gráficos de barras permiten comparar cantidades entre categorías.

Primero contamos cuántas observaciones existen para cada `group`.

##### Edad

In [ ]:
df_edad = df[df['group']=='By Age']

In [ ]:
# Calcular promedio por indicador y grupo de edad
promedio_edad = (
    df_edad
    .groupby(['indicator', 'subgroup'])['value']
    .mean()
      .reset_index()
    .sort_values(['indicator','subgroup','value'], ascending=[True,True,False])

)

promedio_edad

In [ ]:
plt.figure(figsize=(12, 6))


colores = [
    '#7BC8A4',  # verde pastel
    '#7AA6DC',  # azul pastel
    '#f7bd56',   # naranja paster
    '#F6E7A1'   # amarillo pastel
]

sns.barplot(
    data=promedio_edad,
    x='subgroup',
    y='value',
    hue='indicator',
    palette=colores
)

plt.xlabel('Grupo de edad')
plt.ylabel('Porcentaje promedio estimado (%)')
plt.title('Porcentaje promedio por grupo de edad e indicador')


plt.legend(
    title='Indicador',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)


plt.show()

##### Nivel Educativo

In [ ]:
# Filtrar por nivel educativo
df_educacion = df[df['group'] == 'By Education']

# Calcular promedio
promedio_educacion = (
    df_educacion
    .groupby(['indicator', 'subgroup'])['value']
    .mean()
    .reset_index()
)

promedio_educacion

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=promedio_educacion,
    x='value',
    y='subgroup',
    hue='indicator',
    palette=colores
)

plt.xlabel('Porcentaje promedio estimado (%)')
plt.ylabel('Nivel educativo')
plt.title('Porcentaje promedio por nivel educativo e indicador')

plt.xticks(rotation=45)

plt.legend(
    title='Indicador',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.show()

##### Presencia de síntomas de ansiedad y depresión

In [ ]:
# Filtrar por presencia de síntomas de ansiedad/depresión
df_sintomas = df[
    df['group'] == 'By Presence of Symptoms of Anxiety/Depression'
]

# Calcular promedio
promedio_sintomas = (
    df_sintomas
    .groupby(['indicator', 'subgroup'])['value']
    .mean()
    .reset_index()
)

promedio_sintomas

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=promedio_sintomas,
    x='value',
    y='subgroup',
    hue='indicator',
    palette=colores
)

plt.xlabel('Porcentaje promedio estimado (%)')
plt.ylabel('Presencia de síntomas de ansiedad/depresión')
plt.title('Porcentaje promedio por presencia de síntomas e indicador')

plt.xticks(rotation=45)

plt.legend(
    title='Indicador',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.show()

#### Línea del tiempo

Para construir una línea del tiempo necesitamos asegurarnos que la fecha tenga un tipo de dato adecuado.

In [ ]:
df.dtypes

Como existen múltiples observaciones para una misma fecha, primero resumimos la información calculando el promedio de `value` por fecha.

In [ ]:
# Calcular el promedio por fecha e indicador
serie_tiempo = (
    df
    .groupby(['time_period_end_date', 'indicator'])['value']
    .mean()
    .reset_index()
)

serie_tiempo

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=serie_tiempo,
    x='time_period_end_date',
    y='value',
    hue='indicator',
    palette=colores
)

plt.xlabel('Fecha')
plt.ylabel('Porcentaje promedio estimado (%)')
plt.title('Evolución de los indicadores de salud mental')

plt.legend(
    title='Indicador',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.show()

#### Mapa interactivo

In [ ]:
# Filtrar información por estado
df_estados = df[df['group'] == 'By State']

promedio_estados = (
    df_estados
    .groupby('state')['value']
    .mean()
    .reset_index()
)

promedio_estados.head()

In [ ]:
df['indicator'].unique()

In [ ]:
indicador = 'Necesitó terapia pero no la obtuvo'

In [ ]:
df_mapa = df[
    (df['group'] == 'By State') &
    (df['indicator'] == indicador)
]

promedio_estados = (
    df_mapa
    .groupby('state')['value']
    .mean()
    .reset_index()
)

promedio_estados.head()

In [ ]:
abreviaturas = {
    'Alabama':'AL', 'Alaska':'AK', 'Arizona':'AZ', 'Arkansas':'AR',
    'California':'CA', 'Colorado':'CO', 'Connecticut':'CT', 'Delaware':'DE',
    'Florida':'FL', 'Georgia':'GA', 'Hawaii':'HI', 'Idaho':'ID',
    'Illinois':'IL', 'Indiana':'IN', 'Iowa':'IA', 'Kansas':'KS',
    'Kentucky':'KY', 'Louisiana':'LA', 'Maine':'ME', 'Maryland':'MD',
    'Massachusetts':'MA', 'Michigan':'MI', 'Minnesota':'MN', 'Mississippi':'MS',
    'Missouri':'MO', 'Montana':'MT', 'Nebraska':'NE', 'Nevada':'NV',
    'New Hampshire':'NH', 'New Jersey':'NJ', 'New Mexico':'NM',
    'New York':'NY', 'North Carolina':'NC', 'North Dakota':'ND',
    'Ohio':'OH', 'Oklahoma':'OK', 'Oregon':'OR', 'Pennsylvania':'PA',
    'Rhode Island':'RI', 'South Carolina':'SC', 'South Dakota':'SD',
    'Tennessee':'TN', 'Texas':'TX', 'Utah':'UT', 'Vermont':'VT',
    'Virginia':'VA', 'Washington':'WA', 'West Virginia':'WV',
    'Wisconsin':'WI', 'Wyoming':'WY'
}

promedio_estados['codigo'] = promedio_estados['state'].map(abreviaturas)

In [ ]:
fig = px.choropleth(
    promedio_estados,
    locations='codigo',
    locationmode='USA-states',
    color='value',
    scope='usa',
    hover_name='state',
    hover_data={'value': ':.1f', 'codigo': False},
    labels={'value': 'Porcentaje (%)'},
    title='Uso de medicamentos para salud mental por estado'
)

fig.show()